<!-- MUTERBANDUNG_NOTEBOOK_STATUS_V1 -->
# Notebook Status - Pendukung Real-World Verification

| Item | Isi |
|---|---|
| Status | PENDUKUNG_VERIFIKASI |
| Fungsi | Membantu cek status real-world lokasi wisata. |
| Input utama | Queue verifikasi lokasi dan hasil pencarian/verifikasi. |
| Output utama | Catatan verifikasi untuk status aktif, koordinat, atau atribut penting. |
| Aturan pakai | Jalankan hanya untuk queue verifikasi, bukan seluruh dataset sekaligus. |
| Catatan | Keputusan status penting tetap perlu dicatat ringkas di notebook utama. |

---


# MuterBandung Real-world Verification - Google Colab

Notebook ini dibuat untuk verifikasi manual flag real-world dari `manual_realworld_verification_queue.csv`.

Tujuan:

- Upload queue verifikasi real-world.
- Buka Google Search/Maps/official search.
- Tandai flag yang benar-benar terbukti.
- Export `manual_realworld_verification_completed.csv` dan `realworld_manual_apply_ready.csv`.

Aturan aman:

- Jangan set flag `true` tanpa bukti.
- `evidence_url` wajib HTTP/HTTPS untuk baris `approved`.
- Gunakan `flags_to_keep_false` untuk flag yang tidak ditemukan buktinya.
- Notebook ini tidak menulis dataset utama.


In [ ]:
import re
import urllib.parse

import pandas as pd
from IPython.display import display, HTML

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
print("IN_COLAB:", IN_COLAB)


## 1. Upload Queue CSV

Upload file:

```text
Dataset/3_Curated/manual_realworld_verification_queue.csv
```


In [ ]:
QUEUE_PATH = "manual_realworld_verification_queue.csv"

if IN_COLAB:
    uploaded = files.upload()
    if uploaded:
        QUEUE_PATH = next(iter(uploaded.keys()))

df = pd.read_csv(QUEUE_PATH).fillna("")
print("Rows:", len(df))
display(df.head(8))


## 2. Tambahkan Link Pencarian


In [ ]:
def search_url(query, mode="search"):
    encoded = urllib.parse.quote_plus(str(query))
    if mode == "maps":
        return f"https://www.google.com/maps/search/{encoded}"
    return f"https://www.google.com/search?q={encoded}"

for col in ["verified_flags_to_set_true", "flags_to_keep_false", "evidence_url", "reviewer_status", "reviewer_note"]:
    if col not in df.columns:
        df[col] = ""

df["reviewer_status"] = df["reviewer_status"].replace("", "todo").fillna("todo")
df["google_search_url"] = df["location_name"].apply(lambda name: search_url(f"{name} Bandung fasilitas wisata"))
df["google_maps_search_url"] = df["location_name"].apply(lambda name: search_url(f"{name} Bandung", mode="maps"))
df["official_search_url"] = df["location_name"].apply(lambda name: search_url(f"{name} Bandung official website fasilitas"))

display(df[["location_id", "location_name", "priority", "unverified_facility_flags", "unverified_context_flags"]].head(10))


## 3. Batch Kerja Dengan Link Klik


In [ ]:
def clickable(value, label):
    if not isinstance(value, str) or not value.startswith("http"):
        return ""
    return f'<a href="{value}" target="_blank">{label}</a>'

def show_work_batch(start=0, n=12, priority=None):
    work = df.copy()
    if priority:
        work = work[work["priority"].astype(str).str.upper() == priority.upper()]
    batch = work.iloc[start:start+n].copy()
    rows = []
    for _, row in batch.iterrows():
        rows.append({
            "location_id": row.get("location_id", ""),
            "location_name": row.get("location_name", ""),
            "priority": row.get("priority", ""),
            "facility_flags": row.get("unverified_facility_flags", ""),
            "context_flags": row.get("unverified_context_flags", ""),
            "Search": clickable(row.get("google_search_url", ""), "Search"),
            "Maps": clickable(row.get("google_maps_search_url", ""), "Maps"),
            "Official": clickable(row.get("official_search_url", ""), "Official"),
        })
    display(HTML(pd.DataFrame(rows).to_html(escape=False, index=False)))

show_work_batch(start=0, n=12, priority="HIGH")


## 4. Isi Cepat Dengan Dictionary

Format flag memakai pemisah `|`, contoh:

```text
parking_verified|toilet_verified|mushola_verified
```


In [ ]:
manual_updates = [
    # Contoh:
    # {
    #     "location_id": "LOC-001",
    #     "verified_flags_to_set_true": "parking_verified|toilet_verified",
    #     "flags_to_keep_false": "pet_friendly_verified|open_24h_verified",
    #     "evidence_url": "https://...",
    #     "reviewer_status": "approved",
    #     "reviewer_note": "Dicek dari Google Maps/website resmi.",
    # },
]

for update in manual_updates:
    loc_id = str(update.get("location_id", "")).strip()
    if not loc_id:
        continue
    mask = df["location_id"].astype(str) == loc_id
    if not mask.any():
        print("Tidak ditemukan:", loc_id)
        continue
    for key, value in update.items():
        if key == "location_id":
            continue
        if key not in df.columns:
            df[key] = ""
        df.loc[mask, key] = value

print("Approved rows:", int(df["reviewer_status"].astype(str).str.lower().eq("approved").sum()))
display(df[df["reviewer_status"].astype(str).str.lower().eq("approved")].head(20))


## 5. Alternatif: Download Template, Edit, Upload Ulang


In [ ]:
TEMPLATE_PATH = "manual_realworld_verification_colab_template.csv"
df.to_csv(TEMPLATE_PATH, index=False)
print("Template saved:", TEMPLATE_PATH)

if IN_COLAB:
    files.download(TEMPLATE_PATH)


In [ ]:
# Jalankan setelah Anda upload CSV hasil edit.
# Ubah SKIP_UPLOAD_EDITED menjadi False jika ingin memakai file hasil edit.

SKIP_UPLOAD_EDITED = True

if IN_COLAB and not SKIP_UPLOAD_EDITED:
    uploaded = files.upload()
    edited_path = next(iter(uploaded.keys()))
    df = pd.read_csv(edited_path).fillna("")
    print("Loaded edited file:", edited_path, "rows:", len(df))
    display(df.head())


## 6. Validasi Verifikasi


In [ ]:
def is_http_url(value):
    return isinstance(value, str) and re.match(r"^https?://", value.strip(), flags=re.I) is not None

def split_flags(value):
    return {item.strip() for item in str(value or "").split("|") if item.strip()}

def available_flags(row):
    return split_flags(row.get("unverified_facility_flags", "")) | split_flags(row.get("unverified_context_flags", ""))

def validate_row(row):
    status = str(row.get("reviewer_status", "")).strip().lower()
    if status != "approved":
        return False, "not_approved"
    if not is_http_url(row.get("evidence_url", "")):
        return False, "evidence_url_invalid_or_missing"
    true_flags = split_flags(row.get("verified_flags_to_set_true", ""))
    false_flags = split_flags(row.get("flags_to_keep_false", ""))
    if not true_flags and not false_flags:
        return False, "no_flag_decision"
    allowed = available_flags(row)
    unknown = (true_flags | false_flags) - allowed
    if unknown:
        return False, "unknown_flags: " + "|".join(sorted(unknown))
    overlap = true_flags & false_flags
    if overlap:
        return False, "flag_in_true_and_false: " + "|".join(sorted(overlap))
    return True, "ok"

validation = df.apply(validate_row, axis=1, result_type="expand")
df["validation_pass"] = validation[0]
df["validation_note"] = validation[1]

print(df["validation_note"].value_counts().to_string())
display(df[["location_id", "location_name", "verified_flags_to_set_true", "flags_to_keep_false", "evidence_url", "reviewer_status", "validation_pass", "validation_note"]].head(30))


## 7. Export Hasil

Download:

- `manual_realworld_verification_completed.csv`
- `realworld_manual_apply_ready.csv`


In [ ]:
COMPLETED_PATH = "manual_realworld_verification_completed.csv"
APPLY_READY_PATH = "realworld_manual_apply_ready.csv"

df.to_csv(COMPLETED_PATH, index=False)
apply_ready = df[df["validation_pass"] == True].copy()
apply_ready.to_csv(APPLY_READY_PATH, index=False)

print("Completed rows:", len(df))
print("Apply-ready rows:", len(apply_ready))

if IN_COLAB:
    files.download(COMPLETED_PATH)
    files.download(APPLY_READY_PATH)


## 8. Setelah Dari Colab

Simpan `realworld_manual_apply_ready.csv` ke project lokal:

```text
Dataset/3_Curated/realworld_manual_apply_ready.csv
```

Tahap apply ke dataset utama dilakukan setelah audit file hasil Colab.
